# Julie — lab notebook

**Project:** Does a structure predictor notice a mutation?
**Team:** CAIR Structural Biology / Molecular Biophysics, Fall 2026
**Role:** _geometry — superposition, displacement, contact maps_

---

## How to use this notebook

This is your lab notebook, and it is a deliverable in its own right — not scratch space.

- **If you are writing code:** code cells for code, markdown cells for what you were
  trying and what the result meant. A cell with no explanation around it is not a record.
- **If you are not writing code:** markdown cells throughout. Same notebook, same weight.
- **Either way:** at least one literature review, at the bottom of this notebook.

**Findings go in `data/README.md`, not just here.** Row counts, numbering offsets, traps
and the reasons behind decisions belong in that file, in the same commit as the work that
found them. Anything that lives only in a notebook cell is lost the first time someone
restarts a kernel.

## One rule worth stating up front

A null result is a real result on this project. If neither predictor moves — and we have
measured the noise floor — that is a finding we can present. Nobody needs to go looking
for a positive.


## Setup

Run this first. If the imports fail, you are probably not in `notebooks/julie/`.

**Your task brief:** `disagreement_data_analysis/TASK.md` (data in `disagreement_data_analysis/inputs/`).

In [1]:
# Small-matrix numpy work: pin BLAS to one thread.
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import sys
sys.path.append('../../src')

from mutations import parse_variant, apply_variant, assert_variant_set, infer_offset
from geometry import load_ca, superpose, locality_profile, noise_floor

In [2]:
# Smoke test — does the shared code import and work?
seq_ubq = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
print(len(seq_ubq), "residues")
print(assert_variant_set(seq_ubq, ["K48R", "L8A", "I44A"]))

76 residues
{'total': 3, 'passed': 3, 'failed': 0, 'failures': []}


---

## Week 6 — panel frozen, environments working

**Goal this week:** Set up the environment, understand the task and data, and start Step 1 of Task.md (superpose every pair and summarize it).

**What I did:** 
1. Confirmed the environment works and the shared code in /src imports.  
2. Wrote code that loops over every row of pairs.csv, superposes structure B onto structure A on C-alpha using load_ca and superpose, and saves two tables to disagreement_data_analysis/outputs/:
* step1_pair_summary.csv has one row per pair: RMSD, max displacement, mean ΔpLDDT, and ΔpLDDT at the mutated site.
* step1_per_residue.csv has one row per residue per pair: displacement, ΔpLDDT, and an is_site flag.
3. Converted residue numbers from construct to UniProt numbering (using construct_start from constructs.csv) before superposing, so the mutation site from pos_uniprot matches the right residue.
4. Nothing is hardcoded, so the same code should run on the full panel when it arrives.

**What I found:** Sanity check passed.

**Blocked on / questions for the team:** When does the full panel arrive?


In [3]:
# week 6 work

import numpy as np
import pandas as pd
IN = "disagreement_data_analysis/inputs/"
st = pd.read_csv(IN + "tables/structures.csv").set_index("structure_id")
pairs = pd.read_csv(IN + "tables/pairs.csv")
cons = pd.read_csv(IN + "tables/constructs.csv").set_index("protein")

summary = pairs.copy()
rmsd = []
max_displacement = []
mean_delta_plddt = []
site_delta_plddt = []
residue_tables = []

for i in range(len(pairs)):
    start = int(cons.loc[pairs.protein[i], "construct_start"])
    wt = load_ca(IN + st.loc[pairs.structure_a[i], "path"])
    wt.resids = wt.resids + start -1
    mut = load_ca(IN + st.loc[pairs.structure_b[i], "path"])
    mut.resids = mut.resids + start - 1
    d = superpose(ref=wt, mut=mut)
    
    rmsd.append(d.rmsd)
    max_displacement.append(d.max_displacement)
    mean_delta_plddt.append(d.mean_delta_plddt)

    if pairs.kind[i] == "mut_vs_wt":
        site = int(pairs.pos_uniprot[i])
        idx = np.flatnonzero(d.resids == site)
        assert len(idx) == 1, f"site {site} not found in pair {i}"
        site_delta_plddt.append(d.delta_plddt[idx[0]])
    else:
        site_delta_plddt.append(np.nan)
        site = np.nan

    one = pd.DataFrame({
        "pair_id": i,
        "kind": pairs.kind[i],
        "protein": pairs.protein[i],
        "variant": pairs.variant[i],
        "predictor": pairs.predictor[i],
        "resid_uniprot": d.resids,
        "displacement": d.per_residue,
        "delta_plddt": d.delta_plddt,
        "is_site": d.resids == site
    })
    residue_tables.append(one)


summary["rmsd"] = rmsd
summary["max_disp"] = max_displacement
summary["mean_dplddt"] = mean_delta_plddt
summary["site_dplddt"] = site_delta_plddt
per_residue = pd.concat(residue_tables, ignore_index=True)

OUT = "disagreement_data_analysis/outputs/"
os.makedirs(OUT, exist_ok=True)
summary.to_csv(OUT + "step1_pair_summary.csv", index=False)
per_residue.to_csv(OUT + "step1_per_residue.csv", index=False)

per_residue.groupby(["pair_id", "kind"]).is_site.sum()

pair_id  kind     
0        mut_vs_wt    1
1        mut_vs_wt    1
2        mut_vs_wt    1
3        mut_vs_wt    1
4        mut_vs_wt    1
5        mut_vs_wt    1
6        mut_vs_wt    1
7        mut_vs_wt    1
8        mut_vs_wt    1
9        mut_vs_wt    1
10       mut_vs_wt    1
11       mut_vs_wt    1
12       mut_vs_wt    1
13       mut_vs_wt    1
14       mut_vs_wt    1
15       mut_vs_wt    1
16       mut_vs_wt    1
17       mut_vs_wt    1
18       mut_vs_wt    1
19       mut_vs_wt    1
20       mut_vs_wt    1
21       wt_vs_wt     0
22       wt_vs_wt     0
23       wt_vs_wt     0
24       wt_vs_wt     0
25       wt_vs_wt     0
26       wt_vs_wt     0
27       wt_vs_wt     0
28       wt_vs_wt     0
29       wt_vs_wt     0
30       wt_vs_wt     0
Name: is_site, dtype: int64

---

## Week 7 — the noise floor

**Goal this week:** Finish Step 2 of TASK.md: measure the noise floor per protein × predictor.

**What I did:** 
1. Filtered the Step 1 tables to the wt_vs_wt pairs.
2. Grouped by protein × predictor and summarized RMSD (number of pairs, mean, max, 95th percentile) and per-residue displacement (mean, max).
3. Joined the two summaries into one table and saved it as disagreement_data_analysis/outputs/noise_floor.csv.
4. Did not call geometry.noise_floor. The WT pairs were already superposed in Step 1, pairs.csv already defines which folds to compare, and my per-residue table gives the whole displacement distribution, not only the max.

**What I found:** 
1. ESMFold replicate folds are identical, but the floor comes out around 1e-15 Å, not exactly 0, because of rounding in the superposition math. I set anything below a tolerance of 1e-6 Å to 0.0. PDB coordinates only have 3 decimals, so nothing real can be that small.
2. AF2 has a noise floor only for ubq_human in the dev subset, from a single pair (seed 0 vs seed 1). pdz3_rat and ubq_yeast have one AF2 WT fold each, so they have no floor and are absent from noise_floor.csv.

**Blocked on / questions for the team:** 
1. The AF2 floor needs more WT seeds per protein, which the full panel has (5 seeds). Until then Step 4 has no AF2 floor for two of the three proteins.
2. Which column of noise_floor.csv should be "the" floor value for the rendering color scale?

In [4]:
# week 7 work

wt_pairs = summary[summary.kind == "wt_vs_wt"]
wt_pairs[["protein", "predictor", "seed", "rmsd", "max_disp"]]
floor_rmsd = wt_pairs.groupby(["protein", "predictor"]).rmsd.agg(["count", "mean", "max"])
floor_rmsd["p95"] = wt_pairs.groupby(["protein", "predictor"]).rmsd.quantile(0.95)

wt_res = per_residue[per_residue.kind == "wt_vs_wt"]
floor_disp = wt_res.groupby(["protein", "predictor"]).displacement.agg(["mean", "max"])

floor = floor_rmsd.add_prefix("rmsd_").join(floor_disp.add_prefix("disp_"))
floor = floor.rename(columns={"rmsd_count": "n_pairs"})
floor

TOL = 1e-6  # Å; anything smaller is rounding error, not motion
cols = [c for c in floor.columns if c != "n_pairs"]
floor[cols] = floor[cols].mask(floor[cols] < TOL, other=0.0)

floor.to_csv(OUT + "noise_floor.csv")
floor

n_pairs  rmsd_mean  rmsd_max  rmsd_p95  disp_mean  \
protein   predictor                                                       
pdz3_rat  esmfold           3   0.000000  0.000000  0.000000   0.000000   
ubq_human alphafold2        1   0.065145  0.065145  0.065145   0.045805   
          esmfold           3   0.000000  0.000000  0.000000   0.000000   
ubq_yeast esmfold           3   0.000000  0.000000  0.000000   0.000000   

                      disp_max  
protein   predictor             
pdz3_rat  esmfold     0.000000  
ubq_human alphafold2  0.297897  
          esmfold     0.000000  
ubq_yeast esmfold     0.000000

---

## Week 8 — ESMFold mutants + geometry pipeline

**Goal this week:**

**What I did:**

**What I found:**

**Blocked on / questions for the team:**


In [5]:
# week 8 work


---

## Week 9 — AlphaFold2 arm, comparison, renders

**Goal this week:**

**What I did:**

**What I found:**

**Blocked on / questions for the team:**


In [6]:
# week 9 work


---

## Week 10 — figures and the talk

**Goal this week:**

**What I did:**

**What I found:**

**Blocked on / questions for the team:**


In [7]:
# week 10 work


---


## Literature review

> **Format note:** the CAIR handbook defines an official literature review format that
> was not included in the brief handed to this team. Replace this template with the
> official one once you have it. Until then, this covers the same ground.

### Paper

**Citation:**
**Link / DOI:**
**Provenance:** full-text / metadata-only / background-knowledge *(pick one — see below)*

> **Provenance matters on this team.** Label every source as **full-text** (you read the
> paper), **metadata-only** (title and abstract only), or **background-knowledge** (you
> are recalling it, not checking it). This is how we avoid asserting things in a talk
> that nobody actually verified. Our whole project premise is currently metadata-only.

### What question were they asking?

### What did they do?

_Data, method, and the comparison that carries the claim._

### What did they find?

_Numbers, not impressions. If you cannot give a number, say so._

### What is the weakest part?

_Every paper has one. Sample size, a missing control, a benchmark that leaks. For our
project specifically: did they report a run-to-run noise floor? Most do not._

### What does this mean for us?

_One or two sentences. Does it support H1, H0, or neither? Does it change what we
should measure?_

### Questions I still have
